<div align='center'>
<img src='https://img.shields.io/badge/TMDB%205000-Dataset-blue?style=for-the-badge' />
<img src='https://img.shields.io/badge/LangGraph-Agentic%20AI-purple?style=for-the-badge' />
<img src='https://img.shields.io/badge/RandomForest-ML%20Model-green?style=for-the-badge' />
<img src='https://img.shields.io/badge/Google%20Colab-Notebook-orange?style=for-the-badge' />
</div>

---

# 🎬 Movie Matrix AI Pro
### *Film Başarısını Tahmin Eden Agentic AI Sistemi*

---

## 📌 Proje Özeti

Bu notebook, **TMDB 5000 Film Veri Seti** üzerinde uçtan uca bir makine öğrenmesi ve yapay zeka akışı sunar:

| Aşama | Açıklama |
|-------|----------|
| **Veri Temizleme** | Eksik değer analizi, aykırı değer tespiti, JSON parse, feature engineering |
| **EDA** | Dağılım grafikleri, tür/yönetmen/dönem analizleri, interaktif scatter plotlar |
| **Isı Haritası** | 11 özellik arasındaki korelasyon matrisi |
| **Random Forest** | Classifier + Regressor, confusion matrix, feature importance |
| **Lineer Regresyon** | Basit + Ridge, katsayı görselleştirme, residual analizi |
| **Model Metrikleri** | R², MAE, RMSE, 5-fold cross-validation karşılaştırma tablosu |
| **LangGraph Agents** | Orchestrator + Search + Predict + Analyze + Recommend agentic akışı |
| **Film Testi** | TMDB'den gerçek zamanlı veri çekip ML modeliyle tahmin |

```
  Kullanıcı Sorusu
       ↓
  [Orchestrator — LangGraph StateGraph]
       ↓
  ┌────┼──────────────────┐
  ↓    ↓                  ↓
Search  Analyze       Recommend
  ↓
Predict (GBR + RF)
  ↓
Final Response
```

---

> **Dataset:** TMDB 5000 Movies + Credits  
> **Modeller:** GradientBoostingRegressor · RandomForestClassifier · Ridge Regression  
> **Agentic Framework:** LangGraph StateGraph  
> **External API:** TMDB API (Gerçek zamanlı film verisi)

---
## 📦 BÖLÜM 1 — Kurulum & Kütüphaneler

In [ ]:
# ── Tüm bağımlılıkları kur ────────────────────────────────────────────────────
!pip install langgraph langchain-core scikit-learn pandas numpy \
             matplotlib seaborn plotly requests -q

print('✅ Kurulum tamamlandı!')

In [ ]:
import ast, os, re, sys, json, random, warnings
from pathlib import Path
from typing import TypedDict, Optional, List, Literal, Any
from collections import Counter

import numpy as np
import pandas as pd
import requests

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, accuracy_score,
)

warnings.filterwarnings('ignore')

# Görsel ayarlar
plt.rcParams.update({
    'figure.figsize': (13, 6),
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
})
sns.set_palette('husl')

PALETTE = {
    'hit':     '#27ae60',
    'mid':     '#f39c12',
    'risky':   '#e74c3c',
    'blue':    '#3498db',
    'purple':  '#9b59b6',
    'teal':    '#1abc9c',
}

print('✅ Kütüphaneler yüklendi.')
print(f'   pandas {pd.__version__} | numpy {np.__version__}')

---
## 📁 BÖLÜM 2 — Veri Yükleme

> **Seçenek A — Google Drive (önerilen):** `USE_DRIVE = True` yap ve `PROJECT_PATH`'i güncelle.  
> **Seçenek B — Manuel:** `USE_DRIVE = False` yap, CSV dosyalarını sol panelden yükle.

In [ ]:
USE_DRIVE = True   # ← False yaparak manuel yüklemeye geçebilirsin

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_PATH = '/content/drive/MyDrive/movie_matrix_ai'  # ← Kendi yolunu yaz
    os.chdir(PROJECT_PATH)
    print(f'📂 Çalışma dizini: {os.getcwd()}')
else:
    from google.colab import files
    print('⬆️  Yüklenecek dosyalar:')
    print('   • tmdb_5000_movies.csv')
    print('   • tmdb_5000_credits.csv')
    print('   • system_prompt.md')
    print('   • agents/ klasörü (zip olarak)')
    uploaded = files.upload()

In [ ]:
print('📂 CSV dosyaları okunuyor...')
movies_raw  = pd.read_csv('tmdb_5000_movies.csv')
credits_raw = pd.read_csv('tmdb_5000_credits.csv')

print(f'✅ Movies  : {movies_raw.shape[0]:,} satır × {movies_raw.shape[1]} sütun')
print(f'✅ Credits : {credits_raw.shape[0]:,} satır × {credits_raw.shape[1]} sütun')
print('\n--- Movies sütunları ---')
print(list(movies_raw.columns))

---
## 🧹 BÖLÜM 3 — Veri Temizleme

Adımlar:
1. **Merge** (iki CSV birleştirme)
2. **Sıralamaya Göre Eksik Veri Analizi** — sıralı tablo · bar chart · ısı haritası · interaktif panel · önce/sonra
3. **Sıfır bütçe/gelir** filtrelemesi
4. **JSON** sütunlarını parse etme (genres, cast, crew...)
5. **Feature Engineering** (ROI, kâr, dönem, yönetmen ROI, oyuncu gücü)


In [ ]:
# ── 3.1 Merge ────────────────────────────────────────────────────────────────
df_merged = movies_raw.merge(credits_raw, on='title')
print(f'✅ Merge sonrası: {df_merged.shape[0]:,} satır | {df_merged.shape[1]} sütun')
print(f'  Movies sütunları  : {list(movies_raw.columns[:5])}...')
print(f'  Credits sütunları : {list(credits_raw.columns)}')

### 🔍 3.2 Sıralamaya Göre Eksik Veri Analizi

Her iki CSV için ayrı ayrı ve birleşik dataset üzerinde eksik değer analizi yapılmaktadır.  
Sütunlar **eksik değer sayısına göre büyükten küçüğe** sıralanmaktadır.

| Renk | Anlam |
|------|-------|
| 🔴 Kırmızı | Kritik (>%30 eksik) |
| 🟡 Sarı | Orta (%10–30 eksik) |
| 🟢 Yeşil | Az (<%10 eksik) |

In [ ]:
# ── 3.2.1 Üç Dataset İçin Sıralı Eksik Değer Tablosu ─────────────────────

def missing_summary(df_input, name):
    """Sütunları eksik değer sayısına göre büyükten küçüğe sıralar."""
    total   = len(df_input)
    missing = df_input.isnull().sum().sort_values(ascending=False)
    miss_df = pd.DataFrame({
        'Sütun'      : missing.index,
        'Eksik Sayı' : missing.values,
        'Eksik %'    : (missing.values / total * 100).round(2),
        'Dolu Sayı'  : total - missing.values,
        'Dolu %'     : ((total - missing.values) / total * 100).round(2),
        'Veri Tipi'  : [str(df_input[c].dtype) for c in missing.index],
    })
    miss_df['Durum'] = miss_df['Eksik %'].apply(
        lambda x: '🔴 KRİTİK (>%30)' if x > 30
        else ('🟡 ORTA (%10-30)' if x > 10
        else ('🟢 AZ (<%10)' if x > 0 else '✅ Tam'))
    )
    return miss_df

ms_movies  = missing_summary(movies_raw,  "movies_raw")
ms_credits = missing_summary(credits_raw, "credits_raw")
ms_merged  = missing_summary(df_merged,   "df_merged")

print("="*72)
print("  📋 MOVIES CSV — Eksik Değer Tablosu (Sıralı: Eksik Sayı ↓)")
print("="*72)
print(ms_movies[ms_movies["Eksik Sayı"] > 0].to_string(index=False))
print(f"  Toplam satır: {len(movies_raw):,} | Eksik sütun: {(movies_raw.isnull().sum()>0).sum()}")

print("
" + "="*72)
print("  📋 CREDITS CSV — Eksik Değer Tablosu (Sıralı: Eksik Sayı ↓)")
print("="*72)
has_miss = ms_credits[ms_credits["Eksik Sayı"] > 0]
print(has_miss.to_string(index=False) if len(has_miss)>0 else "  ✅ Credits CSV'de eksik değer yok!")

print("
" + "="*72)
print("  📋 MERGE SONRASI — Eksik Değer Tablosu (Sıralı: Eksik Sayı ↓)")
print("="*72)
print(ms_merged[ms_merged["Eksik Sayı"] > 0].to_string(index=False))
print(f"  Toplam satır: {len(df_merged):,} | Toplam sütun: {df_merged.shape[1]}")

In [ ]:
# ── 3.2.2 Renk Kodlu Sıralı Bar Chart (Sayı + Yüzde) ─────────────────────
missing_sorted = df_merged.isnull().sum().sort_values(ascending=False)
missing_sorted = missing_sorted[missing_sorted > 0]

if len(missing_sorted) == 0:
    print("✅ Merge sonrası hiç eksik değer yok.")
else:
    pct = missing_sorted / len(df_merged) * 100
    bar_colors = [
        PALETTE["risky"] if p > 30 else
        PALETTE["mid"]   if p > 10 else
        PALETTE["teal"]
        for p in pct.values
    ]

    fig, axes = plt.subplots(1, 2, figsize=(18, max(5, len(missing_sorted)*0.55+2)))

    # Sol: Sayı bazlı (büyükten küçüğe)
    bars = axes[0].barh(
        missing_sorted.index[::-1], missing_sorted.values[::-1],
        color=bar_colors[::-1], alpha=0.88, edgecolor="white"
    )
    for bar, val, p in zip(bars, missing_sorted.values[::-1], pct.values[::-1]):
        axes[0].text(bar.get_width()+2, bar.get_y()+bar.get_height()/2,
                     f"{int(val):,}  (%{p:.1f})", va="center", fontsize=9.5)
    axes[0].set_xlabel("Eksik Değer Sayısı", fontsize=11)
    axes[0].set_title("📊 Sıralı Eksik Değer Sayısı
(Renk: 🔴>%30 | 🟡%10-30 | 🟢<%10)", fontsize=12)
    axes[0].set_xlim(0, missing_sorted.max() * 1.28)

    # Sağ: Yüzde bazlı
    bars2 = axes[1].barh(
        pct.index[::-1], pct.values[::-1],
        color=bar_colors[::-1], alpha=0.88, edgecolor="white"
    )
    for bar, val in zip(bars2, pct.values[::-1]):
        axes[1].text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
                     f"%{val:.2f}", va="center", fontsize=9.5)
    axes[1].axvline(30, color=PALETTE["risky"], ls="--", lw=1.8, label="Kritik Eşik (%30)")
    axes[1].axvline(10, color=PALETTE["mid"],   ls="--", lw=1.8, label="Orta Eşik (%10)")
    axes[1].set_xlabel("Eksik Değer Yüzdesi (%)", fontsize=11)
    axes[1].set_title("📊 Sıralı Eksik Değer Yüzdesi
(Eşik çizgileri: %10 ve %30)", fontsize=12)
    axes[1].legend(fontsize=9)

    plt.suptitle("🔍 Merge Sonrası Eksik Veri Analizi — Büyükten Küçüğe Sıralı",
                 fontsize=15, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 3.2.3 Eksik Değer Isı Haritası (Sütun × Satır örneği) ─────────────────
missing_cols_sorted = (
    df_merged.isnull().sum()
    .sort_values(ascending=False)
    .pipe(lambda s: s[s > 0])
    .index.tolist()
)

if len(missing_cols_sorted) > 0:
    sample_idx   = df_merged.sample(min(300, len(df_merged)), random_state=42).index
    miss_matrix  = df_merged.loc[sample_idx, missing_cols_sorted].isnull().astype(int)

    fig, ax = plt.subplots(figsize=(max(10, len(missing_cols_sorted)*1.1), 7))
    sns.heatmap(
        miss_matrix.T,
        cmap=["#27ae60", "#e74c3c"],
        cbar=True,
        cbar_kws={"ticks": [0, 1], "label": "0=Dolu | 1=Eksik"},
        yticklabels=missing_cols_sorted,
        xticklabels=False,
        linewidths=0,
        ax=ax
    )
    ax.set_xlabel(f"Örnek Satırlar ({len(sample_idx)} satır)", fontsize=11)
    ax.set_ylabel("Sütun (Eksik Sayısına Göre Sıralı — Çok → Az)", fontsize=11)
    ax.set_title(
        "🗺️  Eksik Değer Isı Haritası — Kırmızı=Eksik, Yeşil=Dolu
"
        "(Sütunlar: eksik sayısı en çok → en az şeklinde sıralı)",
        fontsize=13, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()
else:
    print("✅ Tüm sütunlar tam — ısı haritası oluşturulmadı.")

In [ ]:
# ── 3.2.4 İnteraktif Eksik Veri Paneli (Plotly) ───────────────────────────
ms_only = ms_merged[ms_merged["Eksik Sayı"] > 0].copy().sort_values("Eksik Sayı", ascending=False)

if len(ms_only) > 0:
    bar_clrs = [
        PALETTE["risky"] if p > 30 else
        PALETTE["mid"]   if p > 10 else
        PALETTE["teal"]
        for p in ms_only["Eksik %"]
    ]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            "Eksik Değer Sayısı (Büyükten Küçüğe)",
            "Eksik vs Dolu Oran (%)"
        ]
    )

    fig.add_trace(go.Bar(
        x=ms_only["Sütun"], y=ms_only["Eksik Sayı"],
        marker_color=bar_clrs,
        text=ms_only["Eksik Sayı"].apply(lambda x: f"{x:,}"),
        textposition="outside",
        customdata=ms_only[["Eksik %", "Veri Tipi", "Durum"]].values,
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Eksik Sayı: %{y:,}<br>"
            "Eksik %: %{customdata[0]:.2f}<br>"
            "Tip: %{customdata[1]}<br>"
            "Durum: %{customdata[2]}<extra></extra>"
        ),
        showlegend=False
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=ms_only["Sütun"], y=ms_only["Eksik %"],
        name="Eksik %", marker_color=bar_clrs,
        text=ms_only["Eksik %"].apply(lambda x: f"%{x:.1f}"),
        textposition="inside", showlegend=True
    ), row=1, col=2)
    fig.add_trace(go.Bar(
        x=ms_only["Sütun"], y=ms_only["Dolu %"],
        name="Dolu %", marker_color=PALETTE["blue"],
        opacity=0.4, showlegend=True
    ), row=1, col=2)

    fig.update_layout(
        barmode="stack",
        title_text="🔍 Sıralamaya Göre Eksik Veri Analizi (İnteraktif — üzerine gel)",
        title_font_size=16, template="plotly_white", height=490,
        xaxis_tickangle=-35, xaxis2_tickangle=-35,
    )
    fig.show()
else:
    print("✅ Merge sonrası eksik değer yok — grafik oluşturulmadı.")

In [ ]:
# ── 3.2.5 Aşama Aşama Eksik Veri Değişimi (Önce/Sonra) ───────────────────
df_temp = df_merged[(df_merged["budget"] > 0) & (df_merged["revenue"] > 0)].copy()

stages = {
    "Ham Movies"      : movies_raw,
    "Ham Credits"     : credits_raw,
    "Merge Sonrası"   : df_merged,
    "Bütçe/Gelir Filtre": df_temp,
}

print("="*68)
print("  📊 AŞAMA AŞAMA EKSİK VERİ ÖZETİ (Büyükten Küçüğe Sıralı)")
print("="*68)
print(f"  {"Aşama":<24} {"Satır":>7} {"Sütun":>6} {"Eksik Sütun":>12} {"Toplam Eksik":>13}")
print("  " + "-"*64)
for stage, dfs in stages.items():
    mc = dfs.isnull().sum()
    print(f"  {stage:<24} {len(dfs):>7,} {dfs.shape[1]:>6} {(mc>0).sum():>12} {mc.sum():>13,}")
print("="*68)

# Görsel karşılaştırma
stage_names   = list(stages.keys())
total_miss    = [s.isnull().sum().sum() for s in stages.values()]
miss_col_cnt  = [(s.isnull().sum() > 0).sum() for s in stages.values()]
clrs_s = [PALETTE["risky"], PALETTE["blue"], PALETTE["mid"], PALETTE["teal"]]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

b1 = axes[0].bar(stage_names, total_miss, color=clrs_s, alpha=0.88, edgecolor="white")
for bar, v in zip(b1, total_miss):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                 f"{v:,}", ha="center", fontsize=11, fontweight="bold")
axes[0].set_title("Toplam Eksik Değer Sayısı — Aşamaya Göre")
axes[0].set_ylabel("Toplam Eksik")
axes[0].tick_params(axis="x", rotation=15)

b2 = axes[1].bar(stage_names, miss_col_cnt, color=clrs_s, alpha=0.88, edgecolor="white")
for bar, v in zip(b2, miss_col_cnt):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f"{v} sütun", ha="center", fontsize=11, fontweight="bold")
axes[1].set_title("Eksik Değerli Sütun Sayısı — Aşamaya Göre")
axes[1].set_ylabel("Sütun Sayısı")
axes[1].tick_params(axis="x", rotation=15)

plt.suptitle("🔄 Veri Temizleme Aşamalarında Eksik Veri Değişimi (Önce → Sonra)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Sıfır bütçe/gelir filtresi ──────────────────────────────────────────
n_before = len(df_merged)
df_clean = df_merged[(df_merged['budget'] > 0) & (df_merged['revenue'] > 0)].copy()
print(f'🗑️  Sıfır bütçe/gelir kayıtları kaldırıldı: {n_before - len(df_clean):,}')
print(f'✅ Temiz kayıt: {len(df_clean):,}\n')

# ── 3.5 JSON parse ────────────────────────────────────────────────────────────
def parse_json_col(obj):
    try: return [i['name'] for i in ast.literal_eval(obj)]
    except: return []

def parse_crew_role(obj, job):
    try: return [i['name'] for i in ast.literal_eval(obj) if i.get('job') == job]
    except: return []

def parse_cast(obj, limit=5):
    try: return [i['name'] for i in ast.literal_eval(obj)][:limit]
    except: return []

print('⚙️  JSON sütunları parse ediliyor...')
df_clean['genres_list']    = df_clean['genres'].apply(parse_json_col)
df_clean['keywords_list']  = df_clean['keywords'].apply(parse_json_col)
df_clean['companies_list'] = df_clean['production_companies'].apply(parse_json_col)
df_clean['cast_list']      = df_clean['cast'].apply(lambda x: parse_cast(x, 5))
df_clean['director_list']  = df_clean['crew'].apply(lambda x: parse_crew_role(x, 'Director'))
df_clean['director']       = df_clean['director_list'].apply(lambda x: x[0] if x else 'Unknown')

# ── 3.6 Feature Engineering ──────────────────────────────────────────────────
print('⚙️  Feature engineering...')
df_clean['roi']           = df_clean['revenue'] / df_clean['budget']
df_clean['profit']        = df_clean['revenue'] - df_clean['budget']
df_clean['profit_m']      = df_clean['profit'] / 1e6
df_clean['budget_m']      = df_clean['budget'] / 1e6
df_clean['revenue_m']     = df_clean['revenue'] / 1e6
df_clean['success_class'] = df_clean['roi'].apply(lambda x: 2 if x > 2 else (1 if x > 1 else 0))
df_clean['success_label'] = df_clean['success_class'].map({2: 'Hit', 1: 'Orta', 0: 'Riskli'})
df_clean['release_year']  = pd.to_datetime(df_clean['release_date'], errors='coerce').dt.year
df_clean['decade']        = (df_clean['release_year'] // 10 * 10).astype('Int64').astype(str) + 's'
df_clean['genre_count']   = df_clean['genres_list'].apply(len)

# Yönetmen ortalama ROI
dir_roi = df_clean[df_clean['roi'].between(0, 1000)].groupby('director')['roi'].mean()
df_clean['director_avg_roi'] = df_clean['director'].map(dir_roi).fillna(dir_roi.median())

# Oyuncu gişe gücü
cast_rev = df_clean.explode('cast_list').groupby('cast_list')['revenue'].mean()
df_clean['cast_power'] = df_clean['cast_list'].apply(
    lambda cl: np.mean([cast_rev.get(c, 0) for c in cl]) if cl else 0
)

# Aşırı değerleri kaldır
df_clean = df_clean[df_clean['roi'] < 200].reset_index(drop=True)
df = df_clean.copy()

print(f'\n✅ Feature engineering tamamlandı!')
print(f'   Toplam film     : {len(df):,}')
print(f'   Özellik sayısı  : {df.shape[1]}')
print(f'   Hit filmler     : {(df["success_class"]==2).sum():,}')
print(f'   Orta filmler    : {(df["success_class"]==1).sum():,}')
print(f'   Riskli filmler  : {(df["success_class"]==0).sum():,}')

print('\n' + '='*55)
print('  VERİ TEMİZLEME ÖZETİ')
print('='*55)
print(f'  Ham movies kaydı       : {len(movies_raw):>6,}')
print(f'  Merge sonrası          : {len(df_merged):>6,}')
print(f'  Sıfır bütçe/gelir son. : {len(df_clean):>6,}')
print(f'  ROI outlier sonrası    : {len(df):>6,}')
print('='*55)

---
## 📊 BÖLÜM 4 — Keşifsel Veri Analizi (EDA)

In [ ]:
# ── 4.1 Temel İstatistikler ───────────────────────────────────────────────────
stats_cols   = ['budget_m', 'revenue_m', 'roi', 'vote_average', 'popularity', 'runtime']
stats_labels = ['Bütçe (M$)', 'Gişe (M$)', 'ROI', 'IMDB Puanı', 'Popülarite', 'Süre (dk)']
stats = df[stats_cols].describe().T
stats.index = stats_labels
stats.columns = ['N', 'Ort.', 'Std', 'Min', 'Q1', 'Medyan', 'Q3', 'Max']
print('📋 Sayısal Özet İstatistikler:')
print(stats.round(2).to_string())

In [ ]:
# ── 4.2 Bütçe & Gişe Dağılımı ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, label, color in zip(
    axes,
    ['budget_m', 'revenue_m'],
    ['💰 Bütçe (M$)', '💵 Gişe Geliri (M$)'],
    [PALETTE['blue'], PALETTE['hit']]
):
    ax.hist(df[col], bins=55, color=color, alpha=0.82, edgecolor='white')
    ax.axvline(df[col].median(), color='red', linestyle='--', linewidth=2,
               label=f'Medyan: ${df[col].median():.0f}M')
    ax.axvline(df[col].mean(), color='orange', linestyle='--', linewidth=1.5,
               label=f'Ort: ${df[col].mean():.0f}M')
    ax.set_title(label); ax.set_xlabel('Milyon $'); ax.set_ylabel('Film Sayısı')
    ax.legend()

plt.suptitle('📊 Bütçe ve Gişe Geliri Dağılımı', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.3 ROI Dağılımı & Başarı Pasta ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

roi_plot = df[df['roi'] < 20]
axes[0].hist(roi_plot['roi'], bins=65, color=PALETTE['mid'], alpha=0.85, edgecolor='white')
axes[0].axvline(1, color='red',   linestyle='--', lw=2, label='ROI=1 (Başabaş)')
axes[0].axvline(2, color='green', linestyle='--', lw=2, label='ROI=2 (Hit Eşiği)')
axes[0].set_title('📈 ROI Dağılımı (ROI < 20)')
axes[0].set_xlabel('ROI'); axes[0].set_ylabel('Film Sayısı'); axes[0].legend()

sc = df['success_label'].value_counts()
wedges, texts, auts = axes[1].pie(
    sc.values, labels=sc.index,
    autopct='%1.1f%%', startangle=90,
    colors=[PALETTE['risky'], PALETTE['mid'], PALETTE['hit']],
    explode=(0.05, 0.05, 0.05)
)
[a.set_fontweight('bold') for a in auts]
axes[1].set_title('🎯 Başarı Sınıfı Dağılımı')

plt.suptitle('📊 ROI ve Başarı Analizi', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

for label, cnt in sc.items():
    print(f'  {label:8s}: {cnt:4,} film  ({cnt/len(df)*100:.1f}%)')

In [ ]:
# ── 4.4 En Popüler Türler ─────────────────────────────────────────────────────
all_genres  = [g for genres in df['genres_list'] for g in genres]
genre_cnts  = Counter(all_genres).most_common(15)
genre_df    = pd.DataFrame(genre_cnts, columns=['Tür', 'Film Sayısı'])

fig, ax = plt.subplots(figsize=(13, 7))
pal = sns.color_palette('husl', len(genre_df))
bars = ax.barh(genre_df['Tür'][::-1], genre_df['Film Sayısı'][::-1],
                color=pal[::-1], alpha=0.85)
for bar in bars:
    ax.text(bar.get_width() + 8, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=10)
ax.set_xlabel('Film Sayısı')
ax.set_title('🎭 En Çok Film Çekilen Türler (Top 15)')
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.5 Türe Göre Ortalama Gişe ──────────────────────────────────────────────
top_genre_names = [g[0] for g in genre_cnts[:12]]
genre_revenue = {}
for g in top_genre_names:
    mask = df['genres_list'].apply(lambda gl: g in gl)
    genre_revenue[g] = df[mask]['revenue_m'].mean()

gr_df = pd.DataFrame(list(genre_revenue.items()), columns=['Tür', 'Ort. Gişe (M$)'])
gr_df = gr_df.sort_values('Ort. Gişe (M$)', ascending=True)
median_rev = gr_df['Ort. Gişe (M$)'].median()
colors_rev = [PALETTE['hit'] if v >= median_rev else PALETTE['risky'] for v in gr_df['Ort. Gişe (M$)']]

fig, ax = plt.subplots(figsize=(13, 7))
bars = ax.barh(gr_df['Tür'], gr_df['Ort. Gişe (M$)'], color=colors_rev, alpha=0.85)
for bar in bars:
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width():.0f}M', va='center', fontsize=10)
ax.set_xlabel('Ortalama Gişe (M$)')
ax.set_title('💰 Türe Göre Ortalama Gişe Geliri')
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.6 Yıllık Trend (1980-2016) ─────────────────────────────────────────────
yearly = (
    df[df['release_year'].between(1980, 2016)]
    .groupby('release_year')
    .agg(n=('title','count'), rev=('revenue_m','mean'), bud=('budget_m','mean'))
    .reset_index()
)

fig, ax1 = plt.subplots(figsize=(16, 6))
ax2 = ax1.twinx()

ax1.fill_between(yearly['release_year'], yearly['rev'], alpha=0.15, color=PALETTE['hit'])
ax1.plot(yearly['release_year'], yearly['rev'], color=PALETTE['hit'], lw=2.5, label='Ort. Gişe (M$)')
ax1.plot(yearly['release_year'], yearly['bud'], color=PALETTE['blue'], lw=2, ls='--', label='Ort. Bütçe (M$)')
ax2.bar(yearly['release_year'], yearly['n'], alpha=0.18, color='gray', label='Film Sayısı')

ax1.set_xlabel('Yıl'); ax1.set_ylabel('Milyon $'); ax2.set_ylabel('Film Sayısı')
ax1.set_title('📅 Yıllık Film Üretimi ve Ortalama Gişe (1980-2016)')

l1, lb1 = ax1.get_legend_handles_labels()
l2, lb2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, lb1 + lb2, loc='upper left')
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.7 Bütçe vs Gişe — İnteraktif Scatter (Plotly) ──────────────────────────
fig = px.scatter(
    df.sample(min(900, len(df)), random_state=42),
    x='budget_m', y='revenue_m',
    color='success_label',
    color_discrete_map={'Hit': PALETTE['hit'], 'Orta': PALETTE['mid'], 'Riskli': PALETTE['risky']},
    size='popularity', size_max=22,
    hover_name='title',
    hover_data={'release_year': True, 'vote_average': True, 'roi': ':.2f',
                'director': True, 'budget_m': False, 'revenue_m': False},
    title='💰 Bütçe vs Gişe — Renk: Başarı Sınıfı | Boyut: Popülarite',
    labels={'budget_m': 'Bütçe (M$)', 'revenue_m': 'Gişe (M$)', 'success_label': 'Başarı'},
    opacity=0.72, template='plotly_white'
)
fig.add_shape(type='line', x0=0, y0=0,
              x1=df['budget_m'].quantile(0.99), y1=df['budget_m'].quantile(0.99),
              line=dict(color='gray', dash='dash', width=1.5))
fig.add_annotation(x=230, y=220, text='Başabaş Çizgisi', showarrow=False,
                   font=dict(color='gray', size=11))
fig.show()

In [ ]:
# ── 4.8 IMDB Puanı Analizi ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(df['vote_average'], bins=42, color=PALETTE['purple'], alpha=0.82, edgecolor='white')
axes[0].axvline(df['vote_average'].mean(),   color='red',    ls='--', lw=2, label=f'Ort: {df["vote_average"].mean():.2f}')
axes[0].axvline(df['vote_average'].median(), color='orange', ls='--', lw=2, label=f'Med: {df["vote_average"].median():.2f}')
axes[0].set_title('⭐ IMDB Puan Dağılımı'); axes[0].set_xlabel('Puan'); axes[0].legend()

df['puan_grubu'] = pd.cut(df['vote_average'], bins=[0,5,6,7,8,10], labels=['<5','5-6','6-7','7-8','>8'])
pr = df.groupby('puan_grubu', observed=True)['revenue_m'].median()
cols_p = [PALETTE['risky'], '#e67e22', '#f1c40f', PALETTE['teal'], PALETTE['hit']]
bars = axes[1].bar(pr.index.astype(str), pr.values, color=cols_p, alpha=0.85, edgecolor='white')
for bar in bars:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'${bar.get_height():.0f}M', ha='center', fontsize=10)
axes[1].set_title('💰 Puan Grubuna Göre Medyan Gişe')
axes[1].set_xlabel('IMDB Puan Grubu'); axes[1].set_ylabel('Medyan Gişe (M$)')

plt.suptitle('⭐ IMDB Puanı Analizi', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.9 En Başarılı Yönetmenler (Plotly) ─────────────────────────────────────
dir_stats = (
    df[df['director'] != 'Unknown']
    .groupby('director')
    .agg(n=('title','count'), toplam=('revenue_m','sum'),
         ort_roi=('roi','mean'), ort_puan=('vote_average','mean'))
    .query('n >= 3')
    .nlargest(15, 'toplam')
    .reset_index()
)

fig = px.bar(
    dir_stats, x='toplam', y='director', orientation='h',
    color='ort_roi', color_continuous_scale='RdYlGn',
    hover_data={'n': True, 'ort_puan': ':.1f', 'ort_roi': ':.2f'},
    title='🎬 En Başarılı Yönetmenler (Min 3 Film | Renk: Ort. ROI)',
    labels={'toplam': 'Toplam Gişe (M$)', 'director': 'Yönetmen', 'ort_roi': 'Ort. ROI'},
    template='plotly_white'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# ── 4.10 Film Süresi Analizi ──────────────────────────────────────────────────
rt_clean = df[df['runtime'].between(60, 240)]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(rt_clean['runtime'], bins=45, color=PALETTE['teal'], alpha=0.85, edgecolor='white')
axes[0].axvline(rt_clean['runtime'].mean(), color='red', ls='--', lw=2,
                label=f'Ort: {rt_clean["runtime"].mean():.0f} dk')
axes[0].set_title('⏱️ Film Süresi Dağılımı'); axes[0].set_xlabel('Dakika'); axes[0].legend()

axes[1].scatter(rt_clean['runtime'], rt_clean['revenue_m'], alpha=0.25, s=18, color=PALETTE['teal'])
z = np.polyfit(rt_clean['runtime'], rt_clean['revenue_m'], 1)
xr = np.linspace(60, 240, 100)
axes[1].plot(xr, np.poly1d(z)(xr), 'r-', lw=2.5, label='Trend')
axes[1].set_title('⏱️ Süre vs Gişe Geliri'); axes[1].set_xlabel('Dakika'); axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
# ── 4.11 Döneme Göre Başarı Dağılımı (Stacked Bar) ───────────────────────────
dmask = df['decade'].isin(['1980s', '1990s', '2000s', '2010s'])
ds = df[dmask].groupby(['decade', 'success_label'], observed=True).size().unstack(fill_value=0)
ds_pct = ds.div(ds.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 7))
ds_pct.plot(kind='bar', stacked=True, ax=ax,
            color=[PALETTE['risky'], PALETTE['mid'], PALETTE['hit']],
            alpha=0.85, edgecolor='white', rot=0)
ax.set_title('📅 Döneme Göre Başarı Dağılımı (%)')
ax.set_xlabel('Dönem'); ax.set_ylabel('Yüzde (%)')
ax.legend(title='Başarı', loc='upper right')
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.12 Violin Plot — Başarı Sınıfı × Özellikler ────────────────────────────
viol_cols = [
    ('budget_m', 'Bütçe (M$)'),   ('vote_average', 'IMDB Puanı'),
    ('popularity', 'Popülarite'), ('roi', 'ROI'),
    ('runtime', 'Süre (dk)'),     ('director_avg_roi', 'Yönetmen Ort. ROI'),
]
pal_v = {'Riskli': PALETTE['risky'], 'Orta': PALETTE['mid'], 'Hit': PALETTE['hit']}
order = ['Riskli', 'Orta', 'Hit']

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, (col, label) in zip(axes.flatten(), viol_cols):
    sub = df[df[col].between(df[col].quantile(0.02), df[col].quantile(0.98))]
    sns.violinplot(data=sub, x='success_label', y=col, palette=pal_v,
                   order=order, ax=ax, inner='quartile', alpha=0.8)
    ax.set_title(label); ax.set_xlabel(''); ax.set_ylabel('')

plt.suptitle('🎻 Başarı Sınıfına Göre Özellik Dağılımları', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🔥 BÖLÜM 5 — Isı Haritası (Korelasyon Analizi)

In [ ]:
# ── 5.1 Korelasyon Matrisi ────────────────────────────────────────────────────
corr_cols = [
    'budget_m', 'revenue_m', 'roi', 'profit_m',
    'vote_average', 'vote_count', 'popularity',
    'runtime', 'genre_count', 'director_avg_roi', 'cast_power'
]
corr_labels = [
    'Bütçe (M$)', 'Gişe (M$)', 'ROI', 'Kâr (M$)',
    'IMDB Puanı', 'Oy Sayısı', 'Popülarite',
    'Süre', 'Tür Sayısı', 'Yönetmen ROI', 'Oyuncu Gücü'
]

corr = df[corr_cols].corr()
corr.index   = corr_labels
corr.columns = corr_labels

fig, ax = plt.subplots(figsize=(14, 11))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap=cmap, center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('🔥 Korelasyon Isı Haritası — Film Özellikleri', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# ── 5.2 Gişe ile Korelasyon Sıralaması ───────────────────────────────────────
rev_corr = df[corr_cols].corr()['revenue_m'].drop('revenue_m').sort_values()
rev_corr.index = [corr_labels[corr_cols.index(c)] for c in rev_corr.index]

colors_c = [PALETTE['hit'] if v >= 0 else PALETTE['risky'] for v in rev_corr.values]
fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(rev_corr.index, rev_corr.values, color=colors_c, alpha=0.85)
ax.axvline(0, color='black', lw=0.8)
for bar, val in zip(bars, rev_corr.values):
    ax.text(val + (0.01 if val >= 0 else -0.015),
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10,
            ha='left' if val >= 0 else 'right')
ax.set_title('🔗 Gişe Geliri ile Pearson Korelasyon Katsayıları')
ax.set_xlabel('Korelasyon Katsayısı')
plt.tight_layout(); plt.show()

---
## 🤖 BÖLÜM 6 — Random Forest Analizi

In [ ]:
# ── 6.1 Veri Hazırlama ────────────────────────────────────────────────────────
FEATURE_COLS = [
    'budget', 'runtime', 'popularity', 'vote_count',
    'vote_average', 'genre_count', 'director_avg_roi', 'cast_power'
]
FEATURE_LABELS = [
    'Bütçe', 'Süre', 'Popülarite', 'Oy Sayısı',
    'IMDB Puanı', 'Tür Sayısı', 'Yönetmen ROI', 'Oyuncu Gücü'
]

X        = df[FEATURE_COLS].fillna(0)
y_class  = df['success_class']
y_rev    = df['revenue']

X_tr, X_te, y_tr_c, y_te_c = train_test_split(X, y_class, test_size=0.2, random_state=42, stratify=y_class)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y_rev, test_size=0.2, random_state=42)

print(f'✅ Train: {len(X_tr):,} | Test: {len(X_te):,} | Feature: {X.shape[1]}')

In [ ]:
# ── 6.2 RF Classifier Eğitimi ─────────────────────────────────────────────────
print('🌲 Random Forest Classifier eğitiliyor...')
rf_clf = RandomForestClassifier(n_estimators=250, max_depth=8, random_state=42, n_jobs=-1)
rf_clf.fit(X_tr, y_tr_c)
y_pred_c   = rf_clf.predict(X_te)
rf_clf_acc = accuracy_score(y_te_c, y_pred_c)

print(f'✅ RF Classifier — Test Doğruluğu: %{rf_clf_acc*100:.2f}\n')
print('📊 Sınıflandırma Raporu:')
print(classification_report(y_te_c, y_pred_c, target_names=['Riskli', 'Orta', 'Hit']))

In [ ]:
# ── 6.3 Confusion Matrix ──────────────────────────────────────────────────────
cm     = confusion_matrix(y_te_c, y_pred_c)
cm_pct = cm.astype(float) / cm.sum(axis=1)[:, None] * 100
labels = ['Riskli', 'Orta', 'Hit']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, data, fmt, cmap, title in zip(
    axes, [cm, cm_pct], ['d', '.1f'], ['Blues', 'RdYlGn'],
    ['Confusion Matrix (Sayı)', 'Confusion Matrix (%)']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap=cmap, ax=ax,
                xticklabels=labels, yticklabels=labels, linewidths=0.5)
    ax.set_title(title); ax.set_ylabel('Gerçek'); ax.set_xlabel('Tahmin')

plt.suptitle(f'🤖 Random Forest Classifier — Doğruluk: %{rf_clf_acc*100:.2f}',
             fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 6.4 Feature Importance ────────────────────────────────────────────────────
fi_clf = pd.Series(rf_clf.feature_importances_, index=FEATURE_LABELS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
pal_fi = sns.color_palette('RdYlGn', len(fi_clf))
bars = ax.barh(fi_clf.index, fi_clf.values, color=pal_fi, alpha=0.85)
for bar, val in zip(bars, fi_clf.values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('🌲 Random Forest — Özellik Önem Sıralaması')
plt.tight_layout(); plt.show()

print('\nÖzellik Önemi Sıralaması:')
for name, imp in fi_clf.sort_values(ascending=False).items():
    print(f'  {name:<20}: {imp:.4f}  {"█" * int(imp*200)}')

In [ ]:
# ── 6.5 RF Regressor ─────────────────────────────────────────────────────────
print('🌲 Random Forest Regressor eğitiliyor...')
rf_reg = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_reg.fit(X_tr_r, y_tr_r)
y_rf_pred = rf_reg.predict(X_te_r)

rf_r2   = r2_score(y_te_r, y_rf_pred)
rf_mae  = mean_absolute_error(y_te_r, y_rf_pred) / 1e6
rf_rmse = np.sqrt(mean_squared_error(y_te_r, y_rf_pred)) / 1e6
print(f'✅ RF Regressor — R²: {rf_r2:.4f} | MAE: ${rf_mae:.1f}M | RMSE: ${rf_rmse:.1f}M')

# Gerçek vs Tahmin
act_m = y_te_r.values / 1e6; prd_m = y_rf_pred / 1e6
msk = (act_m < 800) & (prd_m < 800)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].scatter(act_m[msk], prd_m[msk], alpha=0.28, s=22, color=PALETTE['blue'])
mv = max(act_m[msk].max(), prd_m[msk].max())
axes[0].plot([0, mv], [0, mv], 'r--', lw=2, label='Mükemmel Tahmin')
axes[0].set_xlabel('Gerçek Gişe (M$)'); axes[0].set_ylabel('Tahmin (M$)')
axes[0].set_title(f'RF Regressor — Gerçek vs Tahmin (R²={rf_r2:.4f})')
axes[0].legend()

res = act_m - prd_m
axes[1].scatter(prd_m[msk], res[msk], alpha=0.28, s=22, color=PALETTE['mid'])
axes[1].axhline(0, color='red', ls='--', lw=2)
axes[1].set_xlabel('Tahmin (M$)'); axes[1].set_ylabel('Hata (M$)')
axes[1].set_title('RF Regressor — Residual Grafiği')

plt.suptitle('🌲 Random Forest Regressor Performansı', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 📈 BÖLÜM 7 — Lineer Regresyon Analizi

In [ ]:
# ── 7.1 Basit Lineer Regresyon: Bütçe → Gişe ─────────────────────────────────
lr_s = LinearRegression()
lr_s.fit(df[['budget_m']].values, df['revenue_m'].values)
y_lr_s_p = lr_s.predict(df[['budget_m']].values)
r2_s = r2_score(df['revenue_m'].values, y_lr_s_p)

fig, ax = plt.subplots(figsize=(13, 7))
sc = ax.scatter(df['budget_m'], df['revenue_m'],
                c=df['success_class'], cmap='RdYlGn', alpha=0.35, s=22)
bl = np.linspace(0, df['budget_m'].max(), 100)
ax.plot(bl, lr_s.coef_[0]*bl + lr_s.intercept_, 'r-', lw=2.5,
        label=f'Gişe = {lr_s.coef_[0]:.2f}×Bütçe + {lr_s.intercept_:.1f}')
ax.plot([0, df['budget_m'].max()], [0, df['budget_m'].max()], 'k--', alpha=0.35, lw=1.5, label='Başabaş')
ax.set_xlabel('Bütçe (M$)'); ax.set_ylabel('Gişe (M$)')
ax.set_title(f'📈 Basit Lineer Regresyon — R² = {r2_s:.4f}')
ax.legend(); plt.colorbar(sc, ax=ax, label='0=Riskli / 1=Orta / 2=Hit')
plt.tight_layout(); plt.show()
print(f'β₁={lr_s.coef_[0]:.4f}  β₀={lr_s.intercept_:.2f}M  R²={r2_s:.4f}')

In [ ]:
# ── 7.2 Çoklu Lineer Regresyon (Ridge) ───────────────────────────────────────
scaler = StandardScaler()
Xs_tr = scaler.fit_transform(X_tr_r)
Xs_te = scaler.transform(X_te_r)

lr_m = Ridge(alpha=1.0)
lr_m.fit(Xs_tr, y_tr_r)
y_lr_m_p = lr_m.predict(Xs_te)

lr_r2   = r2_score(y_te_r, y_lr_m_p)
lr_mae  = mean_absolute_error(y_te_r, y_lr_m_p) / 1e6
lr_rmse = np.sqrt(mean_squared_error(y_te_r, y_lr_m_p)) / 1e6

print(f'📊 Ridge Regresyon — R²: {lr_r2:.4f} | MAE: ${lr_mae:.1f}M | RMSE: ${lr_rmse:.1f}M')

# Katsayılar
coef = pd.Series(lr_m.coef_, index=FEATURE_LABELS).sort_values()
cclr = [PALETTE['hit'] if v >= 0 else PALETTE['risky'] for v in coef.values]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
bars = axes[0].barh(coef.index, coef.values, color=cclr, alpha=0.85)
axes[0].axvline(0, color='black', lw=0.8)
for bar, val in zip(bars, coef.values):
    axes[0].text(val + (3e5 if val >= 0 else -3e5),
                 bar.get_y()+bar.get_height()/2,
                 f'{val/1e6:.1f}M', va='center', fontsize=9,
                 ha='left' if val >= 0 else 'right')
axes[0].set_title(f'Ridge Katsayıları (R²={lr_r2:.4f})')
axes[0].set_xlabel('Katsayı (standardize)')

# Residuals dağılımı
res_lr = (y_te_r.values - y_lr_m_p) / 1e6
msk_lr = np.abs(res_lr) < 500
axes[1].hist(res_lr[msk_lr], bins=55, color=PALETTE['purple'], alpha=0.82, edgecolor='white')
axes[1].axvline(0, color='red', ls='--', lw=2)
axes[1].set_title('Lineer Regresyon — Hata Dağılımı')
axes[1].set_xlabel('Hata (M$)')

plt.suptitle('📈 Çoklu Lineer Regresyon (Ridge)', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🎯 BÖLÜM 8 — Model Doğruluk Metrikleri & Karşılaştırma

In [ ]:
# ── 8.1 GradientBoosting Regressor ────────────────────────────────────────────
print('⚙️  GradientBoostingRegressor eğitiliyor...')
gbr = GradientBoostingRegressor(n_estimators=250, learning_rate=0.08,
                                  max_depth=5, subsample=0.8, random_state=42)
gbr.fit(X_tr_r, y_tr_r)
y_gbr_p = gbr.predict(X_te_r)

gbr_r2   = r2_score(y_te_r, y_gbr_p)
gbr_mae  = mean_absolute_error(y_te_r, y_gbr_p) / 1e6
gbr_rmse = np.sqrt(mean_squared_error(y_te_r, y_gbr_p)) / 1e6
print(f'✅ GBR — R²: {gbr_r2:.4f} | MAE: ${gbr_mae:.1f}M | RMSE: ${gbr_rmse:.1f}M')

# Ana modeller olarak kaydet
reg_model = gbr
clf_model = rf_clf
print('\n✅ Ana modeller kaydedildi (reg_model=GBR, clf_model=RF).')

In [ ]:
# ── 8.2 5-Fold Cross-Validation ───────────────────────────────────────────────
print('🔄 5-Fold Cross-Validation (RF Classifier)...')
cv = cross_val_score(rf_clf, X, y_class, cv=5, scoring='accuracy', n_jobs=-1)
print(f'   Fold skorları  : {[f"%{s*100:.2f}" for s in cv]}')
print(f'   Ortalama       : %{cv.mean()*100:.2f}')
print(f'   Std Sapma      : %{cv.std()*100:.2f}')

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar([f'Fold {i+1}' for i in range(5)], cv*100,
              color=PALETTE['blue'], alpha=0.85, edgecolor='white')
ax.axhline(cv.mean()*100, color='red', ls='--', lw=2, label=f'Ort: %{cv.mean()*100:.2f}')
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'%{bar.get_height():.2f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 108); ax.set_ylabel('Doğruluk (%)'); ax.legend()
ax.set_title('🔄 5-Fold Cross-Validation — RF Classifier')
plt.tight_layout(); plt.show()

In [ ]:
# ── 8.3 Model Karşılaştırma Tablosu & Grafiği ────────────────────────────────
comp = pd.DataFrame({
    'Model'    : ['Ridge Regresyon', 'Random Forest Reg.', 'Gradient Boosting Reg.'],
    'R²'       : [lr_r2, rf_r2, gbr_r2],
    'MAE (M$)' : [lr_mae, rf_mae, gbr_mae],
    'RMSE (M$)': [lr_rmse, rf_rmse, gbr_rmse],
}).sort_values('R²', ascending=False)

print('='*65)
print('  MODEL KARŞILAŞTIRMA — GİŞE TAHMİNİ (REGRESYON)')
print('='*65)
print(comp.to_string(index=False, float_format='{:.4f}'.format))
print('='*65)
print(f'\n  🏆 EN İYİ MODEL: {comp.iloc[0]["Model"]} (R²={comp.iloc[0]["R²"]:.4f})')
print(f'  🔵 CLASSIFIER   : RF Classifier  Test=%{rf_clf_acc*100:.2f}  CV=%{cv.mean()*100:.2f}±{cv.std()*100:.2f}')

# Görsel karşılaştırma
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors_m = [PALETTE['hit'], PALETTE['blue'], PALETTE['mid']]
models_l = comp['Model'].tolist()

for ax, col, title in zip(
    axes,
    ['R²', 'MAE (M$)', 'RMSE (M$)'],
    ['R² (↑ İyi)', 'MAE — Ort. Mutlak Hata (↓ İyi)', 'RMSE — Kök Kare Hata (↓ İyi)'],
):
    bars = ax.bar(models_l, comp[col], color=colors_m, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, comp[col]):
        lbl = f'{val:.4f}' if col == 'R²' else f'${val:.1f}M'
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                lbl, ha='center', fontsize=11, fontweight='bold')
    if col == 'R²': ax.set_ylim(0, 1.08)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticklabels(models_l, rotation=12, ha='right')

plt.suptitle('🎯 Regresyon Modelleri Karşılaştırması', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 8.4 Feature Importance Karşılaştırması ────────────────────────────────────
fi_gbr = pd.Series(gbr.feature_importances_, index=FEATURE_LABELS).sort_values(ascending=True)
fi_rfc = pd.Series(rf_clf.feature_importances_, index=FEATURE_LABELS).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, fi, title, pal in zip(
    axes, [fi_gbr, fi_rfc],
    ['Gradient Boosting Reg.', 'Random Forest Classifier'],
    ['PiYG', 'coolwarm']
):
    colors = sns.color_palette(pal, len(fi))
    ax.barh(fi.index, fi.values, color=colors, alpha=0.85)
    for i, (_, val) in enumerate(fi.items()):
        ax.text(val + 0.002, i, f'{val:.4f}', va='center', fontsize=10)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Feature Importance')

plt.suptitle('🌲 Model Feature Importance Karşılaştırması', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 8.5 Özet Metrik Paneli ───────────────────────────────────────────────────
print('\n' + '='*65)
print('  📊 MODEL METRİKLERİ ÖZET TABLOSU')
print('='*65)
print(f'\n  SINIFLAMA — Başarı Tahmini (Riskli / Orta / Hit)')
print(f'  ─────────────────────────────────────────────────')
print(f'  Random Forest    Test Acc : %{rf_clf_acc*100:.2f}')
print(f'  Random Forest    CV Acc   : %{cv.mean()*100:.2f} ± {cv.std()*100:.2f}')
print(f'\n  REGRESYON — Gişe Tahmini (Milyon $)')
print(f'  ─────────────────────────────────────────────────')
print(f'  Gradient Boosting  R²: {gbr_r2:.4f}  MAE: ${gbr_mae:.1f}M  RMSE: ${gbr_rmse:.1f}M  ← Ana Model')
print(f'  Random Forest      R²: {rf_r2:.4f}  MAE: ${rf_mae:.1f}M  RMSE: ${rf_rmse:.1f}M')
print(f'  Ridge Regresyon    R²: {lr_r2:.4f}  MAE: ${lr_mae:.1f}M  RMSE: ${lr_rmse:.1f}M')
print('='*65)

---
## 🕸️ BÖLÜM 9 — LangGraph Agentic Sistem Kurulumu

> Bu bölüm, sistem promptunu okuyarak tüm agent'ları tanımlar ve LangGraph StateGraph'ını kurar.

In [ ]:
# ── 9.1 Sistem Promptu Yükle ──────────────────────────────────────────────────
try:
    with open('system_prompt.md', 'r', encoding='utf-8') as f:
        SYSTEM_PROMPT = f.read()
    print('📋 Sistem Promptu Yüklendi:')
    print('='*55)
    print(SYSTEM_PROMPT[:500] + '...')
    print('='*55)
    print(f'✅ Toplam: {len(SYSTEM_PROMPT)} karakter, {len(SYSTEM_PROMPT.splitlines())} satır')
except FileNotFoundError:
    SYSTEM_PROMPT = (
        "Sen Movie Matrix AI'sın. TMDB 5000 dataset üzerinde film analizi,"
        " gişe tahmini ve öneri yapan bir yapay zeka asistanısın."
    )
    print('⚠️  system_prompt.md bulunamadı, varsayılan prompt kullanılıyor.')

In [ ]:
# ── 9.2 TMDB API — Film Arama Fonksiyonu ──────────────────────────────────────
TMDB_KEY = '9dff4a1400db6ba14b347ce0f29b33a8'
BASE_URL = 'https://api.themoviedb.org/3'
POSTER   = 'https://image.tmdb.org/t/p/w500'

def search_movie(query: str, language: str = 'tr-TR') -> dict:
    try:
        res = requests.get(f'{BASE_URL}/search/movie',
            params={'api_key': TMDB_KEY, 'query': query, 'language': language},
            timeout=8).json()
        results = res.get('results', [])
        if not results:
            return {'error': f'"{query}" bulunamadı.', 'found': False}
        movie = results[0]
        detail = requests.get(f'{BASE_URL}/movie/{movie["id"]}',
            params={'api_key': TMDB_KEY, 'language': language, 'append_to_response': 'credits'},
            timeout=8).json()
        crew    = detail.get('credits', {}).get('crew', [])
        cast    = detail.get('credits', {}).get('cast', [])
        pp      = movie.get('poster_path') or detail.get('poster_path')
        return {
            'found': True, 'id': movie['id'],
            'title': movie.get('title', ''),
            'overview': (movie.get('overview') or '')[:500],
            'release_date': movie.get('release_date', 'N/A'),
            'vote_average': movie.get('vote_average', 0),
            'vote_count': movie.get('vote_count', 0),
            'popularity': movie.get('popularity', 0),
            'runtime': detail.get('runtime') or 100,
            'budget': detail.get('budget', 0),
            'revenue': detail.get('revenue', 0),
            'genres': [g['name'] for g in detail.get('genres', [])],
            'directors': [p['name'] for p in crew if p.get('job') == 'Director'],
            'cast': [p['name'] for p in cast[:5]],
            'poster_url': (POSTER + pp) if pp else None,
        }
    except Exception as e:
        return {'error': str(e), 'found': False}



def search_tmdb_people(query: str, department: str = None) -> list:
    """TMDB'den yönetmen/oyuncu arar. department: 'Directing' veya 'Acting'"""
    try:
        res = requests.get(f'{BASE_URL}/search/person',
            params={'api_key': TMDB_KEY, 'query': query, 'language': 'tr-TR'}, timeout=4).json()
        people = []
        for p in res.get('results', [])[:10]:
            if department and p.get('known_for_department', '') != department:
                continue
            people.append(p.get('name', ''))
        return people
    except:
        return []


def fetch_tmdb_person_detail(name: str) -> dict:
    """Kişinin en bilinen yapımlarını TMDB'den çeker."""
    try:
        res = requests.get(f'{BASE_URL}/search/person',
            params={'api_key': TMDB_KEY, 'query': name, 'language': 'tr-TR'}, timeout=4).json()
        if not res.get('results'): return None
        p = res['results'][0]
        known = p.get('known_for', [])
        top_works = []
        for w in known[:3]:
            t = w.get('title') or w.get('name', '')
            y = str(w.get('release_date', w.get('first_air_date', '')))[:4]
            va = w.get('vote_average', 0)
            if t: top_works.append({'title': t, 'year': y, 'vote': va})
        return {'name': p.get('name', name), 'top_works': top_works,
                'popularity': p.get('popularity', 0)}
    except:
        return None

print('✅ TMDB search_movie(), search_tmdb_people(), fetch_tmdb_person_detail() tanımlandı.')

In [ ]:
# ── 9.3 Tahmin Fonksiyonu ─────────────────────────────────────────────────────
GENRE_BUDGET = {
    'Action':180e6,'Adventure':150e6,'Animation':120e6,'Comedy':60e6,
    'Drama':40e6,'Horror':25e6,'Science Fiction':140e6,'Thriller':55e6,
    'Romance':35e6,'Fantasy':120e6,'Crime':50e6,
}

def predict_movie(movie_data: dict) -> dict:
    budget     = movie_data.get('budget', 0)
    runtime    = movie_data.get('runtime', 100) or 100
    popularity = movie_data.get('popularity', 50)
    vote_count = movie_data.get('vote_count', 0)
    vote_avg   = movie_data.get('vote_average', 5.0)
    genres     = movie_data.get('genres', [])
    directors  = movie_data.get('directors', [])
    cast       = movie_data.get('cast', [])
    genre_count = len(genres) if genres else 1

    # Yönetmen ROI
    dr_med = df['director_avg_roi'].median()
    d_name = directors[0] if directors else 'Unknown'
    # TMDB'den yönetmen desteği — dataset dışı yönetmenler için
    d_data = df[df['director'] == d_name]
    dir_roi = d_data['roi'].mean() if len(d_data) > 0 else dr_med
    if np.isnan(dir_roi) or dir_roi <= 0: dir_roi = dr_med

    # Oyuncu gücü
    cast_rev_map = df.explode('cast_list').groupby('cast_list')['revenue'].mean()
    cp = df['cast_power'].median()
    if cast:
        vals = [cast_rev_map.get(c, 0) for c in cast]
        if any(v > 0 for v in vals): cp = np.mean(vals)

    # Bütçe tahmini
    bud_est = budget <= 0
    if bud_est:
        budget = GENRE_BUDGET.get(genres[0] if genres else 'Drama', 70e6)

    X_in = [[budget, runtime, popularity, vote_count, vote_avg, genre_count, dir_roi, cp]]
    rev_pred = float(reg_model.predict(X_in)[0])
    proba    = clf_model.predict_proba(X_in)[0]

    p_fail = float(proba[0])*100
    p_mid  = float(proba[1])*100
    p_hit  = float(proba[2])*100 if len(proba)==3 else (100-p_fail-p_mid)

    roi_p  = rev_pred / budget
    real_rev  = movie_data.get('revenue', 0)

    if real_rev > 0 and budget > 0:
        real_roi = real_rev / budget
        verdict = 'HIT' if real_roi >= 2 else ('ORTA' if real_roi >= 1 else 'RİSKLİ')
    else:
        verdict = 'HIT' if roi_p >= 2 else ('ORTA' if roi_p >= 1 else 'RİSKLİ')

    return dict(
        predicted_revenue=rev_pred, predicted_revenue_m=rev_pred/1e6,
        roi_pred=roi_p, profit_pred=rev_pred-budget, verdict=verdict,
        prob_hit=round(p_hit,1), prob_mid=round(p_mid,1), prob_fail=round(p_fail,1),
        director_avg_roi=round(dir_roi,2), cast_power_m=round(cp/1e6,1),
        budget_estimated=bud_est, budget_used=budget,
        real_revenue=real_rev, real_roi=real_rev/budget if (real_rev>0 and budget>0) else None,
    )

print('✅ predict_movie() tanımlandı.')

In [ ]:
# ── 9.4 Dataset Analiz Fonksiyonları ─────────────────────────────────────────
def analyze_dataset(qt: str, **kwargs) -> dict:
    if qt == 'summary':
        return {
            'total_films'       : len(df),
            'total_revenue_b'   : round(df['revenue'].sum()/1e9, 1),
            'avg_budget_m'      : round(df['budget'].mean()/1e6, 1),
            'avg_revenue_m'     : round(df['revenue'].mean()/1e6, 1),
            'avg_roi'           : round(df['roi'].mean(), 2),
            'hit_rate_pct'      : round((df['success_class']==2).mean()*100, 1),
            'unique_directors'  : int(df['director'].nunique()),
            'avg_score'         : round(df['vote_average'].mean(), 2),
            'top_genre_by_revenue': str(df.explode('genres_list').groupby('genres_list')['revenue'].sum().idxmax()),
            'max_revenue_m'     : round(df['revenue'].max()/1e6, 0),
            'max_revenue_film'  : str(df.loc[df['revenue'].idxmax(), 'title']),
        }
    elif qt == 'top_n':
        metric = kwargs.get('metric', 'revenue')
        n = kwargs.get('n', 10)
        col_map = {'revenue':'revenue','roi':'roi','score':'vote_average','profit':'profit','popularity':'popularity'}
        col = col_map.get(metric, 'revenue')
        top = df.nlargest(n, col)[['title','director','release_date',col]].copy()
        top['release_year'] = top['release_date'].astype(str).str[:4]
        return {'metric': col, 'top_n': n, 'films': top[['title','director','release_year',col]].to_dict('records')}
    elif qt == 'genre':
        df_exp = df.explode('genres_list')
        stats = df_exp.groupby('genres_list').agg(
            n=('title','count'), avg_rev=('revenue','mean'),
            avg_roi=('roi','mean'), hit_rate=('success_class', lambda x: (x==2).mean())
        ).round(3)
        return {'genre_stats': stats.nlargest(10,'avg_rev').to_dict()}
    elif qt == 'decade':
        dec = df[df['decade']!='NaTs'].groupby('decade').agg(
            n=('title','count'), rev_b=('revenue', lambda x: round(x.sum()/1e9,2)),
            avg_roi=('roi','mean'), hit_rate=('success_class', lambda x: round((x==2).mean()*100,1))
        )
        return {'decades': dec.to_dict()}
    return {'error': f'Bilinmeyen analiz türü: {qt}'}


def recommend_movies(genre=None, min_score=7.0, mood=None, n=5, seed=None) -> dict:
    MOOD_MAP = {
        'Action':          ['aksiyon','heyecan','nefes kesen','macera','savaş','patlama'],
        'Comedy':          ['komedi','gülmek','hafif','eğlenceli','komik'],
        'Drama':           ['derin','duygusal','ağlamak','drama','gerçekçi'],
        'Horror':          ['korku','ürkütücü','hayalet'],
        'Science Fiction': ['bilim kurgu','uzay','robot','gelecek','yapay zeka'],
        'Romance':         ['aşk','romantik','sevgi'],
        'Thriller':        ['gerilim','sürpriz','twist','gizem'],
        'Fantasy':         ['fantezi','ejderha','büyü'],
    }
    if mood and not genre:
        ml = mood.lower()
        for g, kws in MOOD_MAP.items():
            if any(k in ml for k in kws):
                genre = g; break
    filtered = df.copy()
    if genre:
        filtered = filtered[filtered['genres_list'].apply(lambda gl: any(g.lower()==genre.lower() for g in gl))]
    filtered = filtered[filtered['vote_average'] >= min_score]
    if filtered.empty:
        return {'found': False, 'message': 'Uygun film bulunamadı.', 'films': []}
    pool = filtered.nlargest(min(50, len(filtered)), 'popularity')
    sel  = pool.sample(n=min(n, len(pool)), random_state=seed)
    films = [
        {'title': r['title'], 'director': r['director'],
         'year': str(r.get('release_date',''))[:4],
         'score': round(r['vote_average'],1), 'genres': ','.join(r['genres_list']),
         'revenue_m': round(r['revenue']/1e6, 0), 'roi': round(r['roi'],2),
         'success': {2:'Hit',1:'Orta',0:'Riskli'}.get(r['success_class'],'—')}
        for _, r in sel.iterrows()
    ]
    return {'found': True, 'total': len(filtered), 'genre': genre or 'Hepsi', 'films': films}

print('✅ analyze_dataset() ve recommend_movies() tanımlandı.')

In [ ]:
# ── 9.5 LangGraph StateGraph Kurulumu ────────────────────────────────────────
try:
    from langgraph.graph import StateGraph, END
    LANGGRAPH_OK = True
    print('✅ LangGraph bulundu!')
except ImportError:
    LANGGRAPH_OK = False
    print('⚠️  LangGraph yüklü değil — rule-based fallback kullanılacak.')


class AgentState(TypedDict):
    user_query: str
    intent: Optional[str]
    movie_title: Optional[str]
    search_result: Optional[dict]
    prediction_result: Optional[dict]
    analysis_result: Optional[dict]
    recommendation_result: Optional[dict]
    final_response: Optional[str]
    error: Optional[str]


def _classify_intent(q: str):
    ql = q.lower()
    rec_sigs = ['ne izlesem','öneri','öner','tavsiye','önerin','akşam','film seç']
    if any(s in ql for s in rec_sigs): return 'recommend', None
    ana_sigs = ['en iyi','top','analiz','istatistik','ortalama','dataset','özet','dönem','tür','yönetmen','hangi']
    if any(s in ql for s in ana_sigs): return 'analyze', None
    words = q.strip().split()
    if 1 <= len(words) <= 6 and not any(w in ql for w in ['öneri','analiz','liste']):
        return 'search_predict', q.strip()
    return 'analyze', None


def node_router(s: AgentState) -> AgentState:
    intent, title = _classify_intent(s['user_query'])
    return {**s, 'intent': intent, 'movie_title': title}

def node_search(s: AgentState) -> AgentState:
    title = s.get('movie_title')
    if not title: return {**s, 'error': 'Film adı belirtilmedi.'}
    return {**s, 'search_result': search_movie(title)}

def node_predict(s: AgentState) -> AgentState:
    sr = s.get('search_result')
    if not sr or not sr.get('found'):
        return {**s, 'error': sr.get('error','Arama verisi yok') if sr else 'Arama yapılmadı'}
    return {**s, 'prediction_result': predict_movie(sr)}

def node_analyze(s: AgentState) -> AgentState:
    ql = s['user_query'].lower()
    qt = 'genre' if 'tür' in ql else 'decade' if 'dönem' in ql else 'top_n' if 'en iyi' in ql else 'summary'
    kwargs = {'n': 10} if qt == 'top_n' else {}
    return {**s, 'analysis_result': analyze_dataset(qt, **kwargs), '_qt': qt}

def node_recommend(s: AgentState) -> AgentState:
    res = recommend_movies(mood=s['user_query'], min_score=7.0, n=5, seed=42)
    return {**s, 'recommendation_result': res}

def node_respond(s: AgentState) -> AgentState:
    if s.get('error'): return {**s, 'final_response': f'❌ {s["error"]}'}
    parts = []
    sr = s.get('search_result')
    if sr and sr.get('found'):
        d = sr
        parts.append(
            f"Film: {d['title']} ({d['release_date'][:4] if d.get('release_date') else '?'})\n"
            f"Puan: {d['vote_average']:.1f}/10 ({d['vote_count']:,} oy)\n"
            f"Bütçe: ${d['budget']/1e6:.0f}M | Gişe: ${d['revenue']/1e6:.0f}M\n"
            f"Türler: {', '.join(d['genres'])}\n"
            f"Yönetmen: {', '.join(d['directors'])}\n"
            f"Oyuncular: {', '.join(d['cast'])}"
        )
    pr = s.get('prediction_result')
    if pr:
        parts.append(
            f"\nTahmini Gişe: ${pr['predicted_revenue_m']:.0f}M\n"
            f"Tahmini ROI : x{pr['roi_pred']:.2f}\n"
            f"Hit: %{pr['prob_hit']:.0f}  Orta: %{pr['prob_mid']:.0f}  Riskli: %{pr['prob_fail']:.0f}\n"
            f"Sonuç: {pr['verdict']}"
        )
    ar = s.get('analysis_result')
    if ar:
        if 'total_films' in ar:
            parts.append(
                f"Dataset: {ar['total_films']:,} film | Ort.Gişe: ${ar['avg_revenue_m']:.0f}M | "
                f"Hit Oranı: %{ar['hit_rate_pct']:.0f} | Rekor: {ar['max_revenue_film']} (${ar['max_revenue_m']:.0f}M)"
            )
        elif 'films' in ar:
            top_txt = '\n'.join(f"  {i+1}. {f['title']} ({f['release_year']}) — {f['revenue']/1e6:.0f}M$"
                                for i, f in enumerate(ar['films'][:5]))
            parts.append(f"Top Filmler:\n{top_txt}")
    rr = s.get('recommendation_result')
    if rr and rr.get('found'):
        lines = [f"Film Önerileri ({rr['genre']}):"]
        for i, f in enumerate(rr['films'], 1):
            lines.append(f"  {i}. {f['title']} ({f['year']}) ⭐{f['score']} {f['success']}")
        parts.append('\n'.join(lines))
    return {**s, 'final_response': '\n\n'.join(parts) or 'Yanıt üretilemedi.'}

def _route(s: AgentState) -> Literal['search','analyze','recommend']:
    i = s.get('intent','analyze')
    return 'search' if i=='search_predict' else ('recommend' if i=='recommend' else 'analyze')


def build_graph():
    if not LANGGRAPH_OK: return None
    g = StateGraph(AgentState)
    for name, fn in [('router',node_router),('search',node_search),('predict',node_predict),
                     ('analyze',node_analyze),('recommend',node_recommend),('respond',node_respond)]:
        g.add_node(name, fn)
    g.set_entry_point('router')
    g.add_conditional_edges('router', _route, {'search':'search','analyze':'analyze','recommend':'recommend'})
    g.add_edge('search','predict'); g.add_edge('predict','respond')
    g.add_edge('analyze','respond'); g.add_edge('recommend','respond')
    g.add_edge('respond', END)
    return g.compile()


def run_agent(query: str) -> str:
    init: AgentState = dict(
        user_query=query, intent=None, movie_title=None,
        search_result=None, prediction_result=None,
        analysis_result=None, recommendation_result=None,
        final_response=None, error=None
    )
    if graph is not None:
        return graph.invoke(init).get('final_response', 'Yanıt yok.')
    s = node_router(init)
    if s['intent'] == 'search_predict':
        s = node_predict(node_search(s))
    elif s['intent'] == 'recommend':
        s = node_recommend(s)
    else:
        s = node_analyze(s)
    return node_respond(s).get('final_response', 'Yanıt yok.')


graph = build_graph()
status = '✅ LangGraph StateGraph hazır!' if graph else '⚠️  Rule-based fallback aktif'
print(status)
print()
print('  Sistem hazır — film arama, analiz ve öneri yapılabilir.')

In [ ]:
# ── 9.6 Sistem Mimarisi Görselleştirme ───────────────────────────────────────
print(f"""
╔══════════════════════════════════════════════════════╗
║          MOVIE MATRIX AI PRO — MİMARİ                ║
╚══════════════════════════════════════════════════════╝

  📋 system_prompt.md  ← Görev tanımı
          ↓
  👤 Kullanıcı Sorusu
          ↓
  🕸️  Orchestrator (LangGraph StateGraph)
          ↓
     [Intent Router]
     ┌────┼─────────────────┐
     ↓    ↓                 ↓
  🔍 Search  📊 Analyze  🍿 Recommend
  (TMDB API) (Dataset)  (Filtre+Ruh Hali)
     ↓
  🤖 Predict
  (GBR + RF Modeli)
     ↓
  ✅ Final Response

Dataset: TMDB 5000 ({:,} film)
ML Modelleri: GradientBoostingReg + RandomForestClassifier
Features: budget · runtime · popularity · vote_count
          vote_average · genre_count · director_avg_roi · cast_power
""".format(len(df)))

---
## 💬 BÖLÜM 10 — Agent ile Konuş

> `query` değişkenini değiştirerek farklı sorgular dene!

In [ ]:
# ── Film arama + tahmin ───────────────────────────────────────────────────────
# 🎯 BU SATIRI DEĞİŞTİR:
query = "Inception"

# ─────────────────────────────────────────────────────────────────────────────
print(f'🔍 Sorgu: {query}')
print('='*60)
response = run_agent(query)
print(response)

In [ ]:
# ── Dataset analizi ───────────────────────────────────────────────────────────
query = "Genel dataset özeti"

print(f'🔍 Sorgu: {query}')
print('='*60)
print(run_agent(query))

In [ ]:
# ── Film önerisi ──────────────────────────────────────────────────────────────
query = "Bu akşam korku filmi izlemek istiyorum"

print(f'🔍 Sorgu: {query}')
print('='*60)
print(run_agent(query))

In [ ]:
# ── İnteraktif Sohbet Modu ────────────────────────────────────────────────────
# 💡 Örnekler:
#   Film:    "Interstellar", "The Dark Knight", "Avengers: Endgame"
#   Analiz:  "En iyi 10 film", "Tür istatistikleri", "Dönem analizi"
#   Öneri:   "Ne izlesem?", "Aksiyon dolu bir şey öner", "Romantik film önerin"

print('🎬 Movie Matrix AI — Film Analiz Asistanı (çıkış için: exit)')
print('='*60)

while True:
    try:
        user_input = input('\n💬 Siz: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n👋 Görüşmek üzere!'); break
    if not user_input: continue
    if user_input.lower() in ('çıkış','exit','quit','q'):
        print('👋 Görüşmek üzere!'); break
    print('\n🤖 AI:')
    print('-'*45)
    print(run_agent(user_input))
    print('-'*45)

---
## 🎬 BÖLÜM 11 — Film Test Et (Manuel Input)

Üç farklı test yöntemi:
- **11A** → Dataset'te film ara
- **11B** → Manuel parametrelerle tahmin
- **11C** → Toplu karşılaştırma

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ── 11A. Dataset'te Film Ara ─────────────────────────────────────────────────
# 🎯 İstediğin filmin adını yaz:
ARAMA_FILMI = "The Dark Knight"
# ══════════════════════════════════════════════════════════════════════════════

results = df[df['title'].str.contains(ARAMA_FILMI, case=False, na=False)]

if results.empty:
    print(f'❌ "{ARAMA_FILMI}" dataset\'te bulunamadı.')
    words = ARAMA_FILMI.split()
    for w in words:
        partial = df[df['title'].str.contains(w, case=False, na=False)]['title'].head(5).tolist()
        if partial: print(f'  "{w}" içeren filmler: {partial}')
else:
    film_row = results.iloc[0]
    print('='*60)
    print(f"  🎬 {film_row['title']}")
    print('='*60)
    for key, val in [
        ('📅 Yıl',       str(film_row.get('release_date',''))[:4]),
        ('🎭 Türler',    ', '.join(film_row['genres_list'])),
        ('🎬 Yönetmen',  film_row.get('director','—')),
        ('⭐ IMDB',      f"{film_row['vote_average']:.1f}/10  ({film_row['vote_count']:,} oy)"),
        ('🌟 Popülarite', f"{film_row['popularity']:.1f}"),
        ('💰 Bütçe',     f"${film_row['budget_m']:.1f}M"),
        ('💵 Gişe',      f"${film_row['revenue_m']:.1f}M"),
        ('📈 ROI',       f"x{film_row['roi']:.2f}"),
        ('💼 Kâr',       f"${film_row['profit_m']:.1f}M"),
        ('🏆 Başarı',    film_row['success_label']),
        ('⏱️  Süre',      f"{film_row.get('runtime','?')} dk"),
    ]:
        print(f'  {key:<15}: {val}')

    # Model tahmini
    fX = film_row[FEATURE_COLS].fillna(0).values.reshape(1,-1)
    pr = reg_model.predict(fX)[0]; pp = clf_model.predict_proba(fX)[0]
    ph = pp[2]*100 if len(pp)==3 else (100-pp[0]*100-pp[1]*100)

    print('\n  🤖 MODEL TAHMİNİ')
    print(f'  Tahmini Gişe  : ${pr/1e6:.1f}M  (Gerçek: ${film_row["revenue_m"]:.1f}M)')
    print(f'  Hata          : %{abs(pr/1e6-film_row["revenue_m"])/film_row["revenue_m"]*100:.1f}')
    print(f'  Hit Olasılık  : %{ph:.1f}  |  Orta: %{pp[1]*100:.1f}  |  Riskli: %{pp[0]*100:.1f}')

In [ ]:
# ── 11A.2 Dataset Filmi — Görsel Panel ────────────────────────────────────────
if not results.empty:
    fr = results.iloc[0]
    fX = fr[FEATURE_COLS].fillna(0).values.reshape(1,-1)
    pp = clf_model.predict_proba(fX)[0]
    ph = pp[2]*100 if len(pp)==3 else 0

    fig = make_subplots(rows=1, cols=2,
        subplot_titles=[f'Başarı Olasılıkları — {fr["title"]}', 'Tür Hit Oranları'])

    fig.add_trace(go.Bar(
        x=['Riskli','Orta','Hit'],
        y=[pp[0]*100, pp[1]*100, ph],
        marker_color=[PALETTE['risky'], PALETTE['mid'], PALETTE['hit']],
        text=[f'%{pp[0]*100:.1f}',f'%{pp[1]*100:.1f}',f'%{ph:.1f}'],
        textposition='outside', showlegend=False
    ), row=1, col=1)

    ghr = {}
    for g in fr['genres_list'][:5]:
        m = df['genres_list'].apply(lambda gl: g in gl)
        ghr[g] = (df[m]['success_class']==2).mean()*100
    if ghr:
        fig.add_trace(go.Pie(
            labels=list(ghr.keys()), values=list(ghr.values()),
            marker_colors=[PALETTE['hit'],PALETTE['blue'],PALETTE['purple'],PALETTE['teal'],PALETTE['mid']]
        ), row=1, col=2)

    fig.update_layout(
        title_text=f'🎬 {fr["title"]} — Analiz Paneli',
        template='plotly_white', height=430
    )
    fig.update_yaxes(range=[0,115], row=1, col=1)
    fig.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ── 11B. MANUEL TAHMİN ───────────────────────────────────────────────────────
# 🎯 Aşağıdaki değerleri değiştirerek istediğin filmi tahmin et!
# ══════════════════════════════════════════════════════════════════════════════

MANUEL = {
    'title'       : 'Avatar: The Way of Water',  # Film adı
    'budget'      : 350_000_000,                  # Bütçe ($) — 0 = tür bazlı otomatik tahmin
    'runtime'     : 192,                          # Süre (dakika)
    'popularity'  : 130.0,                        # Popülarite skoru
    'vote_count'  : 9000,                         # Oy sayısı
    'vote_average': 7.6,                          # IMDB puanı (1-10)

    # Tür — seçenekler: Action, Adventure, Animation, Comedy, Crime,
    #                    Drama, Fantasy, Horror, Romance, Science Fiction,
    #                    Thriller, Mystery, Family, History, Music
    'genres'      : ['Action', 'Adventure', 'Science Fiction'],

    'directors'   : ['James Cameron'],
    'cast'        : ['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver'],
}

# ─────────────────────────────────────────────────────────────────────────────
pred = predict_movie(MANUEL)

print(f"{'='*62}")
print(f"  🎬 {MANUEL['title']} — TAHMİN RAPORU")
print(f"{'='*62}")
for key, val in [
    ('💰 Bütçe',         f"${MANUEL['budget']/1e6:.1f}M{'  (tahmini)' if pred['budget_estimated'] else ''}"),
    ('⏱️  Süre',           f"{MANUEL['runtime']} dk"),
    ('🌟 Popülarite',    f"{MANUEL['popularity']:.1f}"),
    ('⭐ IMDB',          f"{MANUEL['vote_average']}/10"),
    ('🎭 Türler',        ', '.join(MANUEL['genres'])),
    ('🎬 Yönetmen',      f"{MANUEL['directors'][0]} (Ort.ROI x{pred['director_avg_roi']:.2f})"),
    ('🦸 Oyuncular',     ', '.join(MANUEL['cast'])),
]:
    print(f'  {key:<18}: {val}')

print()
print(f"  📈 Tahmini Gişe    : ${pred['predicted_revenue_m']:.1f}M")
print(f"  📈 Tahmini ROI     : x{pred['roi_pred']:.2f}")
print(f"  💼 Tahmini Kâr     : ${pred['profit_pred']/1e6:.1f}M")
print()
print(f"  🔴 Riskli          : %{pred['prob_fail']:.1f}")
print(f"  🟡 Orta            : %{pred['prob_mid']:.1f}")
print(f"  🟢 Hit             : %{pred['prob_hit']:.1f}")
print()
v_icon = {'HIT':'🟢','ORTA':'🟡','RİSKLİ':'🔴'}.get(pred['verdict'],'●')
print(f"  🏆 KARAR           : {v_icon} {pred['verdict']}")
print(f"{'='*62}")

In [ ]:
# ── 11B.2 Manuel Film — Görsel Panel (Plotly) ─────────────────────────────────
pred = predict_movie(MANUEL)
ghr  = {}
for g in MANUEL['genres']:
    m = df['genres_list'].apply(lambda gl: g in gl)
    ghr[g] = (df[m]['success_class']==2).mean()*100

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{'type':'indicator'},{'type':'bar'}],[{'type':'bar'},{'type':'scatter'}]],
    subplot_titles=['Tahmini ROI', 'Başarı Olasılıkları', 'Tür Hit Oranları', 'Dataset Konumu']
)

# ROI gauge
gcolor = PALETTE['hit'] if pred['roi_pred']>=2 else (PALETTE['mid'] if pred['roi_pred']>=1 else PALETTE['risky'])
fig.add_trace(go.Indicator(
    mode='gauge+number+delta', value=pred['roi_pred'],
    delta={'reference':2.0},
    title={'text':'ROI (Hedef: 2.0)'},
    gauge={'axis':{'range':[0,6]},'bar':{'color':gcolor},
           'steps':[{'range':[0,1],'color':'#fee'},{'range':[1,2],'color':'#ffe'},{'range':[2,6],'color':'#efe'}],
           'threshold':{'line':{'color':'red','width':2},'thickness':0.75,'value':2}}
), row=1, col=1)

# Olasılık
fig.add_trace(go.Bar(
    x=['Riskli','Orta','Hit'], y=[pred['prob_fail'],pred['prob_mid'],pred['prob_hit']],
    marker_color=[PALETTE['risky'],PALETTE['mid'],PALETTE['hit']],
    text=[f'%{pred["prob_fail"]:.1f}',f'%{pred["prob_mid"]:.1f}',f'%{pred["prob_hit"]:.1f}'],
    textposition='outside', showlegend=False
), row=1, col=2)

# Tür hit oranı
if ghr:
    fig.add_trace(go.Bar(
        x=list(ghr.keys()), y=list(ghr.values()),
        marker_color=PALETTE['blue'],
        text=[f'%{v:.1f}' for v in ghr.values()],
        textposition='outside', showlegend=False
    ), row=2, col=1)

# Dataset konumu
smp = df.sample(min(500, len(df)), random_state=42)
fig.add_trace(go.Scatter(
    x=smp['budget_m'], y=smp['revenue_m'],
    mode='markers', marker=dict(color='lightgray', size=4, opacity=0.5),
    name='Dataset', showlegend=True
), row=2, col=2)
fig.add_trace(go.Scatter(
    x=[MANUEL['budget']/1e6], y=[pred['predicted_revenue_m']],
    mode='markers+text', text=[MANUEL['title']],
    textposition='top center',
    marker=dict(color=PALETTE['risky'], size=18, symbol='star'),
    name=MANUEL['title'], showlegend=True
), row=2, col=2)

fig.update_layout(
    title_text=f"🎬 {MANUEL['title']} — Kapsamlı Tahmin Paneli",
    template='plotly_white', height=720
)
fig.update_yaxes(range=[0,115], row=1, col=2)
fig.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ── 11C. TOPLU FİLM KARŞILAŞTIRMASI ─────────────────────────────────────────
# 🎯 İstediğin kadar film ekle veya değiştir!
# ══════════════════════════════════════════════════════════════════════════════

FILMLER = [
    {'title':'Düşük Bütçe Korku',    'budget':15e6,  'runtime':92,  'popularity':20,  'vote_count':500,  'vote_average':6.5,'genres':['Horror'],                    'directors':['Unknown'],'cast':[]},
    {'title':'Orta Bütçe Drama',      'budget':50e6,  'runtime':125, 'popularity':48,  'vote_count':2500, 'vote_average':7.4,'genres':['Drama'],                     'directors':['Unknown'],'cast':[]},
    {'title':'Büyük Bütçe Aksiyon',   'budget':200e6, 'runtime':148, 'popularity':100, 'vote_count':8000, 'vote_average':7.8,'genres':['Action','Adventure'],         'directors':['Unknown'],'cast':[]},
    {'title':'Sci-Fi Blockbuster',    'budget':300e6, 'runtime':170, 'popularity':130, 'vote_count':12000,'vote_average':8.2,'genres':['Science Fiction','Action'],   'directors':['James Cameron'],'cast':['Sam Worthington']},
    {'title':'Animasyon Aile Filmi',  'budget':180e6, 'runtime':102, 'popularity':90,  'vote_count':6000, 'vote_average':8.0,'genres':['Animation','Family','Comedy'],'directors':['Unknown'],'cast':[]},
]

# ─────────────────────────────────────────────────────────────────────────────
cast_rev_map = df.explode('cast_list').groupby('cast_list')['revenue'].mean()

rows = []
for film in FILMLER:
    p = predict_movie(film)
    rows.append({
        'Film'         : film['title'],
        'Bütçe (M$)'   : film['budget']/1e6,
        'Tah. Gişe'    : p['predicted_revenue_m'],
        'Tah. ROI'     : p['roi_pred'],
        'Hit %'        : p['prob_hit'],
        'Orta %'       : p['prob_mid'],
        'Riskli %'     : p['prob_fail'],
        'Karar'        : p['verdict'],
    })

df_bulk = pd.DataFrame(rows)
print('📊 TOPLU KARŞILAŞTIRMA')
print(df_bulk.round(1).to_string(index=False))

# ─── Grafik ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
x = range(len(df_bulk))
w = 0.38

axes[0].bar([i-w/2 for i in x], df_bulk['Bütçe (M$)'],  w, label='Bütçe',      color=PALETTE['blue'],   alpha=0.85)
axes[0].bar([i+w/2 for i in x], df_bulk['Tah. Gişe'],    w, label='Tah. Gişe', color=PALETTE['hit'],    alpha=0.85)
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(df_bulk['Film'], rotation=12, ha='right')
axes[0].set_ylabel('Milyon $'); axes[0].set_title('💰 Bütçe vs Tahmini Gişe'); axes[0].legend()

roi_colors = [PALETTE['hit'] if r>=2 else (PALETTE['mid'] if r>=1 else PALETTE['risky']) for r in df_bulk['Tah. ROI']]
axes[1].bar(df_bulk['Film'], df_bulk['Tah. ROI'], color=roi_colors, alpha=0.85)
axes[1].axhline(1, color='gray',  ls='--', lw=1.5, label='Başabaş (ROI=1)')
axes[1].axhline(2, color='green', ls='--', lw=1.5, label='Hit Eşiği (ROI=2)')
for i, v in enumerate(df_bulk['Tah. ROI']):
    axes[1].text(i, v+0.04, f'x{v:.2f}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_xticklabels(df_bulk['Film'], rotation=12, ha='right')
axes[1].set_ylabel('ROI'); axes[1].set_title('📈 Tahmini ROI'); axes[1].legend()

plt.suptitle('🎬 Toplu Film Karşılaştırması', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ── 11D. TMDB'DEN CANLI FİLM ÇEK + TAHMİN ET ────────────────────────────────
# 🎯 Bu hücre TMDB API'ye bağlanır ve gerçek zamanlı veri çeker!
# ══════════════════════════════════════════════════════════════════════════════

TMDB_ARAMA = "Oppenheimer"  # ← Buraya istediğin filmi yaz!

# ─────────────────────────────────────────────────────────────────────────────
print(f'🌐 TMDB\'den "{TMDB_ARAMA}" aranıyor...')
tmdb_data = search_movie(TMDB_ARAMA)

if not tmdb_data.get('found'):
    print(f'❌ {tmdb_data.get("error")}')
else:
    d = tmdb_data
    print('='*60)
    print(f"  🎬 {d['title']}")
    print('='*60)
    for key, val in [
        ('📅 Yıl',        str(d['release_date'])[:4]),
        ('🎭 Türler',     ', '.join(d['genres'])),
        ('🎬 Yönetmen',   ', '.join(d['directors']) if d['directors'] else '—'),
        ('🦸 Oyuncular',  ', '.join(d['cast'])),
        ('⭐ IMDB',       f"{d['vote_average']:.1f}/10  ({d['vote_count']:,} oy)"),
        ('🌟 Popülarite', f"{d['popularity']:.1f}"),
        ('⏱️  Süre',       f"{d['runtime']} dk"),
        ('💰 Bütçe',      f"${d['budget']/1e6:.0f}M" if d['budget'] else 'Bilinmiyor'),
        ('💵 Gişe',       f"${d['revenue']/1e6:.0f}M" if d['revenue'] else 'Bilinmiyor'),
    ]:
        print(f'  {key:<16}: {val}')

    if d.get('overview'):
        print(f"\n  📖 Özet: {d['overview'][:200]}...")

    # Tahmin
    pred = predict_movie(d)
    v_icon = {'HIT':'🟢','ORTA':'🟡','RİSKLİ':'🔴'}.get(pred['verdict'],'●')
    print(f'\n  🤖 MODEL TAHMİNİ')
    print(f'  Tahmini Gişe   : ${pred["predicted_revenue_m"]:.1f}M')
    print(f'  Tahmini ROI    : x{pred["roi_pred"]:.2f}')
    print(f'  Hit: %{pred["prob_hit"]:.1f}  Orta: %{pred["prob_mid"]:.1f}  Riskli: %{pred["prob_fail"]:.1f}')
    print(f'  KARAR          : {v_icon} {pred["verdict"]}')
    if pred['real_revenue'] and pred['real_revenue'] > 0:
        print(f'  Gerçek Gişe    : ${pred["real_revenue"]/1e6:.0f}M  (ROI: x{pred["real_roi"]:.2f})')
        print(f'  Tahmin Hatası  : %{abs(pred["predicted_revenue_m"]-pred["real_revenue"]/1e6)/(pred["real_revenue"]/1e6)*100:.1f}')

In [ ]:
# ── 11D.2 TMDB Filmi — Görsel Analiz ──────────────────────────────────────────
if tmdb_data.get('found'):
    pred = predict_movie(tmdb_data)
    d = tmdb_data

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[
            'Başarı Olasılıkları',
            'Tür Hit Oranları',
            'Dataset\'teki Konumu'
        ]
    )

    fig.add_trace(go.Bar(
        x=['Riskli','Orta','Hit'],
        y=[pred['prob_fail'],pred['prob_mid'],pred['prob_hit']],
        marker_color=[PALETTE['risky'],PALETTE['mid'],PALETTE['hit']],
        text=[f'%{pred["prob_fail"]:.1f}',f'%{pred["prob_mid"]:.1f}',f'%{pred["prob_hit"]:.1f}'],
        textposition='outside', showlegend=False
    ), row=1, col=1)

    ghr = {}
    for g in d['genres'][:5]:
        m = df['genres_list'].apply(lambda gl: g in gl)
        if m.sum() > 0: ghr[g] = (df[m]['success_class']==2).mean()*100
    if ghr:
        fig.add_trace(go.Bar(
            x=list(ghr.keys()), y=list(ghr.values()),
            marker_color=PALETTE['blue'],
            text=[f'%{v:.1f}' for v in ghr.values()],
            textposition='outside', showlegend=False
        ), row=1, col=2)

    smp = df.sample(min(500,len(df)), random_state=42)
    fig.add_trace(go.Scatter(
        x=smp['budget_m'], y=smp['revenue_m'],
        mode='markers', marker=dict(color='lightgray',size=4,opacity=0.5),
        name='Dataset Filmleri', showlegend=True
    ), row=1, col=3)
    b_used = d['budget'] if d['budget']>0 else pred['budget_used']
    fig.add_trace(go.Scatter(
        x=[b_used/1e6], y=[pred['predicted_revenue_m']],
        mode='markers+text', text=[d['title']],
        textposition='top center',
        marker=dict(color=PALETTE['risky'],size=18,symbol='star'),
        name=d['title'], showlegend=True
    ), row=1, col=3)

    fig.update_layout(
        title_text=f"🌐 {d['title']} (TMDB Canlı) — Analiz Paneli",
        template='plotly_white', height=450
    )
    fig.update_yaxes(range=[0,115], row=1, col=1)
    fig.show()

---

## 📌 Hızlı Referans Tablosu

| Hücre | Değişken | Amaç |
|-------|----------|------|
| **11A** | `ARAMA_FILMI` | Dataset'ten film ara ve ML tahmini yap |
| **11B** | `MANUEL` dict | Kendi parametrelerinle tahmin et |
| **11C** | `FILMLER` list | Birden fazla filmi yan yana karşılaştır |
| **11D** | `TMDB_ARAMA` | TMDB'den canlı veri çek + tahmin et |
| **10** | `query` str | Agent sohbet modu |

### Başarı Sınıfları
| Sınıf | ROI | Açıklama |
|-------|-----|----------|
| 🔴 **Riskli** | < 1.0 | Zarar eden film |
| 🟡 **Orta** | 1.0 – 2.0 | Kârlı ama sıradan |
| 🟢 **Hit** | > 2.0 | Büyük gişe başarısı |

### Kullanılabilir Film Türleri
`Action` · `Adventure` · `Animation` · `Comedy` · `Crime` · `Drama` · `Fantasy` · `Horror` · `Music` · `Mystery` · `Romance` · `Science Fiction` · `Thriller` · `Family` · `History`

---
*Movie Matrix AI Pro — TMDB 5000 Dataset + LangGraph + GradientBoosting + RandomForest*